# 02 — Filter tomato polygons

After notebook 01, ensure `configs/paths.local.yaml` has `landiq.crop_column` and `landiq.tomato_values`.

This notebook loads the LandIQ file, filters rows, and writes `data/derived/landiq_tomato/` (adjust paths as needed).

In [ ]:
from pathlib import Path

import geopandas as gpd
import yaml

from src.landiq.filter_tomato import filter_tomato

cfg_path = Path("../configs/paths.local.yaml")
if not cfg_path.is_file():
    cfg_path = Path("../configs/paths.example.yaml")
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
lc = cfg["landiq"]
raw = Path("../") / cfg["data"]["raw_landiQ"]
year = lc.get("year")
search = (raw / str(year)) if year is not None else raw
glob_pat = lc.get("shapefile_glob", "*.shp")
recursive = lc.get("shapefile_recursive", True)
shps = sorted(search.rglob(glob_pat) if recursive else search.glob(glob_pat))
assert shps, f"No .shp under {search.resolve()} — extract ZIPs and/or set landiq.year"
assert len(shps) == 1, f"Expected one .shp match, got {len(shps)}. Set landiq.year or tighten glob."
gdf = gpd.read_file(shps[0])
tomato = filter_tomato(gdf, lc["crop_column"], lc["tomato_values"])
out_dir = Path("../") / cfg["data"]["derived_tomato"]
out_dir.mkdir(parents=True, exist_ok=True)
out_gpkg = out_dir / "landiq_tomato.gpkg"
tomato.to_file(out_gpkg, driver="GPKG")
print(len(tomato), "features ->", out_gpkg.resolve())